# Milestone 4

Multiple Choice Fine-tuning with BERT + LoRA

Email: **23f3000717@ds.study.iitm.ac.in**

In [ ]:
# Install (Kaggle)
!pip -q install transformers peft datasets accelerate

In [ ]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (AutoTokenizer,
                          AutoModelForMultipleChoice,
                          Trainer,
                          TrainingArguments)
from peft import LoraConfig, TaskType, get_peft_model

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")


## Q1 Label Encoding

In [ ]:
label_map={"A":0,"B":1,"C":2,"D":3,"E":4}
train["labels"]=train["answer"].map(label_map)
print("Answer:",train.loc[150,"labels"])

## Q2 Prompt-Option Formatting

In [ ]:
formatted = str(train.loc[0,"prompt"]) + " [SEP] " + str(train.loc[0,"B"])
print("Length:",len(formatted))

## Tokenizer

In [ ]:
tokenizer=AutoTokenizer.from_pretrained("bert-base-uncased")

## Q3 Single-row Tokenization

In [ ]:
choices=[str(train.loc[0,"prompt"])+" [SEP] "+str(train.loc[0,c]) for c in ["A","B","C","D","E"]]
enc=tokenizer(choices,padding="max_length",truncation=True,max_length=128,return_tensors="pt")
input_ids=enc["input_ids"].unsqueeze(0)
attention_mask=enc["attention_mask"].unsqueeze(0)
print(input_ids.shape)
print("Answer:",input_ids.shape[1])

## Q4 Batch Tokenization

In [ ]:
batch_ids=[]
batch_masks=[]
for i in range(16):
    ch=[str(train.loc[i,"prompt"])+" [SEP] "+str(train.loc[i,c]) for c in ["A","B","C","D","E"]]
    e=tokenizer(ch,padding="max_length",truncation=True,max_length=128,return_tensors="pt")
    batch_ids.append(e["input_ids"])
    batch_masks.append(e["attention_mask"])
batch_ids=torch.stack(batch_ids)
print(batch_ids.shape)
print("Answer:",batch_ids.numel())

## Q5 Multiple-choice logits

In [ ]:
model=AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")
out=model(input_ids=input_ids,attention_mask=attention_mask)
print(out.logits.shape)
print("Answer:",out.logits.shape[1])

## Q6 Loss Tensor

In [ ]:
labels=torch.tensor([train.loc[0,"labels"]])
out=model(input_ids=input_ids,attention_mask=attention_mask,labels=labels)
print(out.loss.shape)
print("Answer:",out.loss.ndim)

## Q7 LoRA

In [ ]:
config=LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query","value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS
)
lora_model=get_peft_model(model,config)
trainable=sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print("Trainable:",trainable)

## Q8 Hugging Face Dataset

In [ ]:
rows=[]
for _,row in train.head(100).iterrows():
    ch=[str(row["prompt"])+" [SEP] "+str(row[c]) for c in ["A","B","C","D","E"]]
    e=tokenizer(ch,padding="max_length",truncation=True,max_length=128)
    rows.append({
        "input_ids":e["input_ids"],
        "attention_mask":e["attention_mask"],
        "labels":int(row["labels"])
    })
ds=Dataset.from_list(rows)
print(len(ds[0]["input_ids"]))

## Q9 Tiny Fine-tuning

In [ ]:
args=TrainingArguments(
    output_dir="mcq_lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    report_to="none"
)
trainer=Trainer(
    model=lora_model,
    args=args,
    train_dataset=ds.select(range(32))
)
trainer.train()
print("global_step:",trainer.state.global_step)

## Q10 Probability of Option E

In [ ]:
out=lora_model(input_ids=input_ids,attention_mask=attention_mask)
probs=torch.softmax(out.logits,dim=-1)
print("Option E probability:",round(probs[0,4].item(),4))